In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

## Downloading and loading dataset from HuggingFace

In [3]:
ds_train = load_dataset("coastalcph/lex_glue", "unfair_tos", split="train")
ds_val = load_dataset("coastalcph/lex_glue", "unfair_tos", split="validation")
ds_test = load_dataset("coastalcph/lex_glue", "unfair_tos", split="test")

In [4]:
ds_train.to_pandas().to_csv("../../data/raw/train.csv", index=False)
ds_val.to_pandas().to_csv("../../data/raw/val.csv", index=False)
ds_test.to_pandas().to_csv("../../data/raw/test.csv", index=False)

In [5]:
train = pd.read_csv("../../data/raw/train.csv")
val = pd.read_csv("../../data/raw/val.csv")
test = pd.read_csv("../../data/raw/test.csv")

## Converting multi-labels list to one-hot encoded labels


In [ ]:
import re

from sklearn.preprocessing import MultiLabelBinarizer


def parse_labels(s):
    return [int(x) for x in re.findall(r"\d+", s)]

for d in (train, val, test):
    d['labels'] = d['labels'].apply(parse_labels)

NAMES = ['Limitation of liability',
 'Unilateral termination',
 'Unilateral change',
 'Content removal',
 'Contract by using',
 'Choice of law',
 'Jurisdiction',
 'Arbitration']

mlb = MultiLabelBinarizer(classes=list(range(8)))
Y_train = mlb.fit_transform(train["labels"])
Y_val = mlb.transform(val["labels"])
Y_test = mlb.transform(test["labels"])

## Removing duplicates and conflicts

In [10]:
for d in (train, val, test):
    d["text"] = d["text"].fillna("").apply(lambda s: re.sub(r"\s+", " ", s).strip()) # Whitespcae normalization
    d["lab_key"] = d["labels"].apply(lambda x: tuple(sorted(x)))

In [11]:
print("dups inside:", train["text"].duplicated().sum(), val["text"].duplicated().sum(), test["text"].duplicated().sum())

va, te = set(val["text"]), set(test["text"])
print("train-val overlap:", train["text"].isin(va).sum())
print("train-test overlap:", train["text"].isin(te).sum())

conflict = train.groupby("text")["lab_key"].nunique()
print("conflicting labels:", (conflict > 1).sum())

dups inside: 105 14 7
train-val overlap: 22
train-test overlap: 36
conflicting labels: 1


In [12]:
n0 = len(train)
train = train.drop_duplicates(subset=["text", "lab_key"])
train = train[~train["text"].isin(va | te)].reset_index(drop=True)
print(n0, "->", len(train))

Y_train = mlb.fit_transform(train["labels"])

5532 -> 5380


In [13]:
print("dups inside:", train["text"].duplicated().sum(), val["text"].duplicated().sum(), test["text"].duplicated().sum())

va, te = set(val["text"]), set(test["text"])
print("train-val overlap:", train["text"].isin(va).sum())
print("train-test overlap:", train["text"].isin(te).sum())

conflict = train.groupby("text")["lab_key"].nunique()
print("conflicting labels:", (conflict > 1).sum())

dups inside: 1 14 7
train-val overlap: 0
train-test overlap: 0
conflicting labels: 1


In [14]:
bad = conflict[conflict > 1].index
train = train[~train["text"].isin(bad)].reset_index(drop=True)
Y_train = mlb.fit_transform(train["labels"])

## Saving cleaned data

In [ ]:
from pathlib import Path

out_dir = Path("../../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

def build(df, Y):
    out = pd.DataFrame(Y, columns=NAMES)
    out.insert(0, "text", df["text"].values)
    out['is_unfair'] = (Y.sum(1) > 0).astype(int)
    return out

build(train, Y_train).to_csv(out_dir / "train.csv", index=False)
build(val, Y_val).to_csv(out_dir / "val.csv", index=False)
build(test, Y_test).to_csv(out_dir / "test.csv", index=False)